# Rice Disease Detection: AI AgriData Intelligence Race (TELEPATI 8.0)

Notebook ini memuat seluruh alur pengembangan model deteksi penyakit padi, mulai dari dataset mentah resmi sampai inference dan perhitungan skor.

| | |
|---|---|
| Tugas | Object detection 11 kelas (10 penyakit + Healthy), citra 640×640 |
| Data | Dataset resmi Agritech Data (format COCO, split train/valid/test) |
| Model | Ultralytics YOLO, dibangun dari file `.yaml` dan dilatih dari nol |
| Metrik | F1-Score makro pada split test resmi |
| Platform | Google Colab, GPU T4 |

**Mode eksekusi**

| `MODE` | Yang dijalankan | Kegunaan |
|---|---|---|
| `"full"` | Bagian 1 sampai 10 | Reproduksi lengkap: data mentah → training → evaluasi |
| `"inference_only"` | Bagian 1 sampai 3, lalu 11 | Reproduksi skor dari weights rilis, tanpa training |

**Kepatuhan aturan**

- Seluruh pemrosesan data dimulai dari zip dataset mentah dan dikerjakan di notebook ini.
- Training hanya memakai split **train**. Split **valid** dipakai untuk memilih checkpoint. Split **test** hanya dipakai di bagian 10 (sekali) dan bagian 11.
- Arsitektur dibangun dari file `.yaml` dengan `pretrained=False`. Tidak ada bobot eksternal yang dipakai untuk training.
- Seed tetap (`SEED = 42`) dan mode deterministik di setiap tahap.

## 0. Konfigurasi

Semua parameter yang perlu diubah ada di sel ini. Sel lain tidak perlu disunting.

- `RUN_ID` memilih konfigurasi training dari tabel `RUNS` di bagian 7.
- `WEIGHTS_URL`, `WEIGHTS_SHA256`, dan `E2_CONF` diisi setelah model final dirilis.
- `RUN_FINAL_TEST` dibiarkan `False` sampai model final dipilih dari hasil valid.
- `REFINE_WEIGHTS = True` menjalankan bagian 9b (model soup dan kalibrasi bias per kelas) untuk run `RUN_ID`.

In [32]:
MODE = "audit"  # "full" | "inference_only" | "audit"
RUN_ID = "B_yolo26s_s42"

DATA_ZIP = "/content/drive/MyDrive/telepati/data/telepati-agridata-raw.zip"  # file zip, atau folder hasil ekstrak
DATA_ZIP_SHA256 = "797e47749f91f4403e778dd0150716da1cd94cddbecee2ec56cf5545c50cec50"  # zip referensi tim

RUNS_DIR = "/content/drive/MyDrive/telepati/runs"
WORK_DIR = "/content/work"

WEIGHTS_URL = "https://github.com/DimasPanca/telepati-agridata/releases/download/v1.0/telepati_agridata_v1.0.pt"
WEIGHTS_SHA256 = "cdd815e88a22b6d9d6ee205dba7e51944e9c563177f6bdab01d1d984cc373b67"
E2_CONF = 0.26  # dipilih dari valid (bagian 9)

SEED = 42
IMGSZ = 640
EVAL_BATCH = 16
RUN_FINAL_TEST = False
REFINE_WEIGHTS = False
CLIP_BOXES_FOR_EXPORT = True

## 1. Lingkungan

Memasang versi library yang dipin bila versi terpasang berbeda (Colab, Kaggle, maupun lokal), menghubungkan Google Drive (khusus Colab), lalu mengunci seed.

Catatan: saat training dengan AMP, Ultralytics mengunduh model kecil hanya untuk *memeriksa* dukungan mixed precision pada GPU. Bobot tersebut tidak dipakai sebagai inisialisasi; model tetap dibangun dari `.yaml`.

In [33]:
import subprocess
import sys
from importlib import metadata

IN_COLAB = "google.colab" in sys.modules
ULTRALYTICS_VERSION = "8.4.184"

try:
    installed_version = metadata.version("ultralytics")
except metadata.PackageNotFoundError:
    installed_version = None
if installed_version != ULTRALYTICS_VERSION:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
import hashlib
import json
import math
import os
import random
import shutil
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

WORK_DIR = Path(WORK_DIR)
RUNS_DIR = Path(RUNS_DIR)
EXTRACT_DIR = WORK_DIR / "raw"
YOLO_DIR = WORK_DIR / "yolo"
SPLITS = ("train", "valid", "test")

print(f"python      {sys.version.split()[0]}")
print(f"torch       {torch.__version__}")
print(f"ultralytics {ultralytics.__version__}")
print(f"device      {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

python      3.13.16
torch       2.11.0+cu130
ultralytics 8.4.184
device      Tesla T4


## 2. Dataset mentah

Notebook menerima dua bentuk zip:

- zip resmi dari panitia, yang membungkus split dalam satu folder dan menyertakan file metadata macOS (`__MACOSX`, `._*`, `.DS_Store`);
- zip kerja tim, yang langsung berisi `train/`, `valid/`, dan `test/`;
- folder hasil ekstrak, untuk platform yang mengekstrak arsip secara otomatis saat diunggah (misalnya dataset Kaggle). Isi `DATA_ZIP` dengan path folder tersebut.

File metadata macOS dilewati saat ekstraksi, lalu folder yang berisi ketiga split dicari otomatis. Ekstraksi dianggap selesai hanya jika setiap gambar yang tercantum di file COCO benar-benar ada, sehingga ekstraksi yang sempat terputus diulang dari awal. Checksum dibandingkan dengan zip referensi tim sebagai informasi; validitas isi dipastikan lewat jumlah gambar dan anotasi per split.

In [35]:
EXPECTED_COUNTS = {
    "train": {"images": 10133, "annotations": 20163},
    "valid": {"images": 2106, "annotations": 4888},
    "test": {"images": 1059, "annotations": 2670},
}
MACOS_METADATA = ("__MACOSX/", "/._", ".DS_Store")


def sha256sum(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_dataset_root(extract_dir: Path) -> Path | None:
    roots = [p.parent.parent for p in extract_dir.rglob("_annotations.coco.json")
             if p.parent.name == "train" and "__MACOSX" not in p.parts]
    return min(roots, key=lambda p: len(p.parts)) if roots else None


def is_complete(root: Path | None) -> bool:
    if root is None:
        return False
    for split in SPLITS:
        annotation_file = root / split / "_annotations.coco.json"
        if not annotation_file.exists():
            return False
        with open(annotation_file, encoding="utf-8") as fh:
            expected = {image["file_name"] for image in json.load(fh)["images"]}
        if not expected <= {p.name for p in (root / split).iterdir()}:
            return False
    return True


def prepare_dataset(zip_path: Path, extract_dir: Path, reference_sha256: str) -> Path:
    if zip_path.is_dir():
        root = find_dataset_root(zip_path)
        if not is_complete(root):
            raise RuntimeError(f"Folder train/, valid/, dan test/ yang lengkap tidak ditemukan di {zip_path}")
        print(f"Memakai dataset yang sudah terekstrak di {root}")
        return root

    if reference_sha256:
        actual = sha256sum(zip_path)
        if actual == reference_sha256:
            print("Checksum sama dengan zip referensi tim.")
        else:
            print(f"Checksum berbeda dari zip referensi tim ({actual[:12]}...); isi divalidasi lewat jumlah data.")

    root = find_dataset_root(extract_dir) if extract_dir.exists() else None
    if is_complete(root):
        print(f"Dataset sudah terekstrak di {root}")
        return root

    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.infolist() if not any(tag in "/" + m.filename for tag in MACOS_METADATA)]
        zf.extractall(extract_dir, members=members)

    root = find_dataset_root(extract_dir)
    if not is_complete(root):
        raise RuntimeError(f"Folder train/, valid/, dan test/ yang lengkap tidak ditemukan di {DATA_ZIP}")
    print(f"Dataset diekstrak ke {root}")
    return root


def load_coco(split: str) -> dict:
    with open(RAW_DIR / split / "_annotations.coco.json", encoding="utf-8") as fh:
        return json.load(fh)


RAW_DIR = prepare_dataset(Path(DATA_ZIP), EXTRACT_DIR, DATA_ZIP_SHA256)
coco = {split: load_coco(split) for split in SPLITS}

for split, expected in EXPECTED_COUNTS.items():
    n_images, n_annotations = len(coco[split]["images"]), len(coco[split]["annotations"])
    assert n_images == expected["images"], f"{split}: {n_images} gambar"
    assert n_annotations == expected["annotations"], f"{split}: {n_annotations} anotasi"
    print(f"{split:<6} {n_images:>6} gambar  {n_annotations:>6} anotasi")

Checksum berbeda dari zip referensi tim (6bb7fb68cd97...); isi divalidasi lewat jumlah data.
Dataset sudah terekstrak di /content/work/raw/Telepati 8.0 Datasets
train   10133 gambar   20163 anotasi
valid    2106 gambar    4888 anotasi
test     1059 gambar    2670 anotasi


## 3. Pemetaan 21 kategori → 11 kelas canonical

File COCO bawaan memiliki 21 kategori: 18 nama kelas dengan variasi penulisan dan 3 supercategory (`Leaf-blight`, `Rice-Leaf-Diseasee`, `paddy`) yang tidak memiliki bounding box. Pemetaan dilakukan berdasarkan nama kategori, lalu divalidasi agar tidak ada anotasi yang tertinggal.

Urutan kelas mengikuti urutan alfabet pada halaman regulasi dan dipakai konsisten di training, evaluasi, dan inference.

In [36]:
CLASS_NAMES = [
    "Bacterial leaf blight",
    "Bacterial panicle blight",
    "Blast",
    "Brown spot",
    "False smut",
    "Healthy",
    "Leaf roller",
    "Leaf scald",
    "Narrow brown",
    "Sheath blight",
    "Tungro",
]

RAW_TO_CANONICAL = {
    "Bacterial leaf blight": "Bacterial leaf blight",
    "Bacterial panicle Blight": "Bacterial panicle blight",
    "Blast": "Blast",
    "Leaf blast": "Blast",
    "Infected Blast": "Blast",
    "BrownSpot": "Brown spot",
    "Brown spot": "Brown spot",
    "False-Smut": "False smut",
    "Healthy Rice Leaf": "Healthy",
    "Healthy Rice beads": "Healthy",
    "Healthy": "Healthy",
    "healthy": "Healthy",
    "Leaf-roller": "Leaf roller",
    "Leaf Scald": "Leaf scald",
    "Leaf scald": "Leaf scald",
    "Narrow brown": "Narrow brown",
    "Sheath Blight": "Sheath blight",
    "Rice-Tungro": "Tungro",
}
SUPERCATEGORIES = {"Leaf-blight", "Rice-Leaf-Diseasee", "paddy"}
CLASS_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}


def to_frames(data: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Returns (images, boxes); boxes use canonical class ids and COCO xywh in pixels.
    category_names = {c["id"]: c["name"] for c in data["categories"]}
    unknown = set(category_names.values()) - set(RAW_TO_CANONICAL) - SUPERCATEGORIES
    if unknown:
        raise ValueError(f"Kategori tanpa pemetaan: {sorted(unknown)}")

    images = pd.DataFrame(data["images"])[["id", "file_name", "width", "height"]]
    rows = []
    for ann in data["annotations"]:
        raw_name = category_names[ann["category_id"]]
        if raw_name in SUPERCATEGORIES:
            continue
        x, y, w, h = ann["bbox"]
        rows.append((ann["image_id"], CLASS_INDEX[RAW_TO_CANONICAL[raw_name]], x, y, w, h))
    boxes = pd.DataFrame(rows, columns=["image_id", "class_id", "x", "y", "w", "h"])
    boxes = boxes.merge(images, left_on="image_id", right_on="id").drop(columns="id")
    return images, boxes


frames = {split: to_frames(coco[split]) for split in SPLITS}

for split in SPLITS:
    _, boxes = frames[split]
    assert len(boxes) == EXPECTED_COUNTS[split]["annotations"], split

class_counts = pd.DataFrame(
    {split: frames[split][1]["class_id"].value_counts().sort_index() for split in SPLITS}
).set_axis(CLASS_NAMES)
class_counts.loc["Total"] = class_counts.sum()
class_counts

,train,valid,test
Bacterial leaf blight,476,125,59
Bacterial panicle blight,528,45,27
Blast,4149,947,535
Brown spot,5010,1461,840
False smut,852,86,43
Healthy,2374,424,184
Leaf roller,812,87,44
Leaf scald,1438,390,222
Narrow brown,222,75,36
Sheath blight,1762,488,269


## 4. Pembersihan anotasi

Pembersihan hanya diterapkan pada data **train**. Label valid dan test dipakai apa adanya (hanya dipetakan ke 11 kelas) supaya skor yang kita hitung sama dengan perhitungan juri.

| Aturan | Tindakan |
|---|---|
| Koordinat di luar batas gambar | Dipotong ke batas gambar |
| Sisi box < 2 px | Dibuang (luas 1 sampai 3 px², hampir pasti label rusak) |
| Box identik dalam satu gambar | Disisakan satu |
| Box seluas gambar penuh | Dipertahankan (gaya label sebagian sumber data) |
| Gambar tanpa anotasi | Dipertahankan sebagai gambar latar |

In [37]:
MIN_BOX_SIDE = 2.0


def clip_boxes(boxes: pd.DataFrame) -> pd.DataFrame:
    out = boxes.copy()

    x1 = out["x"].clip(
        lower=0,
        upper=out["width"],
    )
    y1 = out["y"].clip(
        lower=0,
        upper=out["height"],
    )

    x2 = (out["x"] + out["w"]).clip(
        lower=0,
        upper=out["width"],
    )
    y2 = (out["y"] + out["h"]).clip(
        lower=0,
        upper=out["height"],
    )

    out["x"] = x1
    out["y"] = y1
    out["w"] = x2 - x1
    out["h"] = y2 - y1

    return out


def audit_split(
    split: str,
    boxes: pd.DataFrame,
) -> tuple[dict, pd.Series]:
    raw = boxes.reset_index(drop=True).copy()
    clipped = clip_boxes(raw)

    x2 = raw["x"] + raw["w"]
    y2 = raw["y"] + raw["h"]

    out_of_bounds = (
        (raw["x"] < 0)
        | (raw["y"] < 0)
        | (x2 > raw["width"])
        | (y2 > raw["height"])
        | (x2 < 0)
        | (y2 < 0)
        | (raw["x"] > raw["width"])
        | (raw["y"] > raw["height"])
    )

    changed_by_clip = (
        ~np.isclose(raw["x"], clipped["x"])
        | ~np.isclose(raw["y"], clipped["y"])
        | ~np.isclose(raw["w"], clipped["w"])
        | ~np.isclose(raw["h"], clipped["h"])
    )

    non_positive = (
        (raw["w"] <= 0)
        | (raw["h"] <= 0)
    )

    tiny = (
        (raw["w"] < MIN_BOX_SIDE)
        | (raw["h"] < MIN_BOX_SIDE)
    )

    full_image = (
        (raw["x"] <= 0)
        & (raw["y"] <= 0)
        & (x2 >= raw["width"])
        & (y2 >= raw["height"])
    )

    duplicate = raw.duplicated(
        subset=[
            "image_id",
            "class_id",
            "x",
            "y",
            "w",
            "h",
        ],
        keep="first",
    )

    summary = {
        "split": split,
        "annotations": len(raw),
        "out_of_bounds": int(out_of_bounds.sum()),
        "changed_by_clip": int(changed_by_clip.sum()),
        "non_positive_box": int(non_positive.sum()),
        "tiny_box_side_lt_2px": int(tiny.sum()),
        "full_image_box": int(full_image.sum()),
        "exact_duplicate_box": int(duplicate.sum()),
    }

    class_names = raw["class_id"].map(
        dict(enumerate(CLASS_NAMES))
    )

    class_counts = (
        class_names
        .value_counts()
        .reindex(CLASS_NAMES, fill_value=0)
    )
    class_counts.name = split

    return summary, class_counts


def clipping_impact_report(
    split: str,
    boxes: pd.DataFrame,
) -> pd.DataFrame:
    raw = boxes.reset_index(drop=True).copy()
    clipped = clip_boxes(raw)

    changed = (
        ~np.isclose(raw["x"], clipped["x"])
        | ~np.isclose(raw["y"], clipped["y"])
        | ~np.isclose(raw["w"], clipped["w"])
        | ~np.isclose(raw["h"], clipped["h"])
    )

    report = raw.loc[
        changed,
        [
            "image_id",
            "file_name",
            "class_id",
            "x",
            "y",
            "w",
            "h",
            "width",
            "height",
        ],
    ].copy()

    report["split"] = split
    report["class_name"] = report["class_id"].map(
        dict(enumerate(CLASS_NAMES))
    )

    report["clipped_x"] = clipped.loc[
        changed, "x"
    ].to_numpy()

    report["clipped_y"] = clipped.loc[
        changed, "y"
    ].to_numpy()

    report["clipped_w"] = clipped.loc[
        changed, "w"
    ].to_numpy()

    report["clipped_h"] = clipped.loc[
        changed, "h"
    ].to_numpy()

    return report


def audit_dataset(
    frames: dict,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    class_tables = []
    impact_tables = []

    for split in SPLITS:
        _, boxes = frames[split]

        summary, class_counts = audit_split(
            split,
            boxes,
        )

        impact = clipping_impact_report(
            split,boxes,
        )

        summary_rows.append(summary)
        class_tables.append(class_counts)
        impact_tables.append(impact)

    summary_table = (
        pd.DataFrame(summary_rows)
        .set_index("split")
    )

    class_table = pd.concat(
        class_tables,
        axis=1,
    ).fillna(0).astype(int)

    impact_table = pd.concat(
        impact_tables,
        ignore_index=True,
    )

    return (
        summary_table,
        class_table,
        impact_table,
    )


def prepare_export_boxes(
    split: str,
    boxes: pd.DataFrame,
) -> pd.DataFrame:
    """
    Menyiapkan box untuk format YOLO.
    Dataset mentah tidak diubah.
    """
    if not CLIP_BOXES_FOR_EXPORT:
        export_boxes = boxes.copy()
    else:
        export_boxes = clip_boxes(boxes)

    invalid = (
        (export_boxes["w"] <= 0)
        | (export_boxes["h"] <= 0)
    )

    if invalid.any():
        raise ValueError(
            f"{split}: ditemukan {int(invalid.sum())} "
            "box dengan lebar/tinggi <= 0 setelah preprocessing."
        )

    return export_boxes


def clean_train_boxes(
    boxes: pd.DataFrame,
) -> pd.DataFrame:
    """
    Cleaning terbatas untuk train saja.
    Dataset mentah tetap tidak diubah.
    """
    clipped = prepare_export_boxes(
        "train",
        boxes,
    )

    too_small = (
        (clipped["w"] < MIN_BOX_SIDE)
        | (clipped["h"] < MIN_BOX_SIDE)
    )

    kept = clipped[~too_small]

    deduped = kept.drop_duplicates(
        subset=[
            "image_id",
            "class_id",
            "x",
            "y",
            "w",
            "h",
        ]
    )

    print(
        f"train: {len(boxes)} box -> "
        f"{len(deduped)} box; "
        f"tiny: {int(too_small.sum())}; "
        f"duplikat: {len(kept) - len(deduped)}"
    )

    return deduped


train_images, raw_train_boxes = frames["train"]
valid_images, raw_valid_boxes = frames["valid"]
test_images, raw_test_boxes = frames["test"]

audit_summary, audit_class_counts, clipping_report = (
    audit_dataset(frames)
)

AUDIT_DIR = RUNS_DIR / "dataset_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

audit_summary.to_csv(
    AUDIT_DIR / "annotation_audit_summary.csv"
)

audit_class_counts.to_csv(
    AUDIT_DIR / "annotation_class_counts.csv"
)

clipping_report.to_csv(
    AUDIT_DIR / "clipping_impact_details.csv",
    index=False,
)

print("=== RINGKASAN AUDIT ===")
display(audit_summary)

print("=== DISTRIBUSI KELAS ===")
display(audit_class_counts)

print("=== DETAIL BOX YANG BERUBAH ===")
display(clipping_report.head(20))

print(f"Hasil audit: {AUDIT_DIR}")


train_boxes = clean_train_boxes(raw_train_boxes)
valid_boxes = prepare_export_boxes(
    "valid",
    raw_valid_boxes,
)
test_boxes = prepare_export_boxes(
    "test",
    raw_test_boxes,
)



=== RINGKASAN AUDIT ===


,annotations,out_of_bounds,changed_by_clip,non_positive_box,tiny_box_side_lt_2px,full_image_box,exact_duplicate_box
split,,,,,,,
train,20163,427,425,0,22,11,7
valid,4888,34,34,0,8,5,1
test,2670,24,24,0,2,1,1


=== DISTRIBUSI KELAS ===


,train,valid,test
class_id,,,
Bacterial leaf blight,476,125,59
Bacterial panicle blight,528,45,27
Blast,4149,947,535
Brown spot,5010,1461,840
False smut,852,86,43
Healthy,2374,424,184
Leaf roller,812,87,44
Leaf scald,1438,390,222
Narrow brown,222,75,36


=== DETAIL BOX YANG BERUBAH ===


,image_id,file_name,class_id,x,y,w,h,width,height,split,class_name,clipped_x,clipped_y,clipped_w,clipped_h
0,1,IMG-20240927-WA0175_jpg.rf.f5aa86568a488fd6465...,5,1,234,639.446,406.059,640,640,train,Healthy,1,234,639.000,406.000
1,36,20240915_150435_jpg.rf.f7a2cd70416a986dde0149e...,6,135,286,505.107,259.748,640,640,train,Leaf roller,135,286,505.000,259.748
2,40,IMG-20241014-WA0235-1-_jpg.rf.f872aa57229d705c...,4,10,165,630.462,369.349,640,640,train,False smut,10,165,630.000,369.349
3,42,IMG-20241020-WA0129_jpg.rf.f84e8e7152fb7572c9e...,2,121,1,332.308,639.385,640,640,train,Blast,121,1,332.308,639.000
4,66,IMG-20241012-WA0315_jpg.rf.f93e02fbad32e791d0a...,1,213,260,427.077,242.129,640,640,train,Bacterial panicle blight,213,260,427.000,242.129
5,97,IMG-20240927-WA0162_jpg.rf.fc1e1c5d0dc1533ec07...,5,184,211,456.285,245.333,640,640,train,Healthy,184,211,456.000,245.333
6,100,IMG_20241020_084802_403_jpg.rf.fc7621b34ce6336...,5,193,94,340.417,546.111,640,640,train,Healthy,193,94,340.417,546.000
7,101,IMG-20240927-WA0007_jpg.rf.fc6f2f3042e014963ca...,5,119,1,351.852,639.444,640,640,train,Healthy,119,1,351.852,639.000
8,127,IMG-20241016-WA0146_jpg.rf.fe05022e84b3e9a2c40...,4,117,1,354.074,639.444,640,640,train,False smut,117,1,354.074,639.000
9,134,IMG-20241014-WA0306_jpg.rf.fe6fc92dff47ce20b7c...,4,285,72,172.800,568.355,640,640,train,False smut,285,72,172.800,568.000


Hasil audit: /content/drive/MyDrive/telepati/runs/dataset_audit
train: 20163 box -> 20134 box; tiny: 22; duplikat: 7


## 5. Konversi ke format YOLO

Setiap gambar ditautkan (hard link, atau disalin bila tidak didukung) ke struktur `images/<split>` dan labelnya ditulis ke `labels/<split>` dalam format `class cx cy w h` ternormalisasi.

Untuk run dengan penyeimbangan kelas, gambar train diulang menurut *repeat factor sampling*: kelas yang muncul di sedikit gambar mendapat faktor `r_c = max(1, sqrt(t / f_c))`, dan setiap gambar diulang sebanyak faktor terbesar dari kelas di dalamnya. Pembulatan memakai RNG dengan seed tetap. Hanya data train yang dipakai untuk menghitung faktor ini.

In [38]:
def link_or_copy(src: Path, dst: Path) -> None:
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def yolo_lines(
    image_boxes: pd.DataFrame,
) -> list[str]:
    lines = []

    for row in image_boxes.itertuples(index=False):
        cx = (row.x + row.w / 2) / row.width
        cy = (row.y + row.h / 2) / row.height
        nw = row.w / row.width
        nh = row.h / row.height

        values = np.array(
            [cx, cy, nw, nh],
            dtype=float,
        )

        if not np.all(np.isfinite(values)):
            raise ValueError(
                f"Nilai label tidak finite: {values}"
            )

        if not (
            0 <= cx <= 1
            and 0 <= cy <= 1
            and 0 < nw <= 1
            and 0 < nh <= 1
        ):
            raise ValueError(
                "Label YOLO di luar rentang valid: "
                f"class={row.class_id}, values={values}"
            )

        lines.append(
            f"{row.class_id} "
            f"{cx:.6f} "
            f"{cy:.6f} "
            f"{nw:.6f} "
            f"{nh:.6f}"
        )

    return lines


def export_split(split: str, images: pd.DataFrame, boxes: pd.DataFrame, out_root: Path,
                 repeats: dict[str, int] | None = None) -> int:
    image_dir, label_dir = out_root / "images" / split, out_root / "labels" / split
    for folder in (image_dir, label_dir):
        shutil.rmtree(folder, ignore_errors=True)
        folder.mkdir(parents=True)

    boxes_by_file = dict(tuple(boxes.groupby("file_name")))
    empty = boxes.iloc[0:0]
    written = 0
    for file_name in sorted(images["file_name"]):
        lines = "\n".join(yolo_lines(boxes_by_file.get(file_name, empty)))
        stem, suffix = Path(file_name).stem, Path(file_name).suffix
        for k in range((repeats or {}).get(file_name, 1)):
            name = stem if k == 0 else f"{stem}__rep{k}"
            link_or_copy(RAW_DIR / split / file_name, image_dir / f"{name}{suffix}")
            (label_dir / f"{name}.txt").write_text(lines)
            written += 1
    return written


def repeat_factors(images: pd.DataFrame, boxes: pd.DataFrame, threshold: float, seed: int) -> dict[str, int]:
    classes_per_image = boxes.groupby("file_name")["class_id"].agg(set)
    image_freq = {c: sum(c in s for s in classes_per_image) / len(images) for c in range(len(CLASS_NAMES))}
    class_factor = {c: max(1.0, math.sqrt(threshold / f)) if f > 0 else 1.0 for c, f in image_freq.items()}

    rng = np.random.default_rng(seed)
    repeats = {}
    for file_name in sorted(images["file_name"]):
        classes = classes_per_image.get(file_name, set())
        factor = max((class_factor[c] for c in classes), default=1.0)
        repeats[file_name] = int(factor) + int(rng.random() < factor - int(factor))
    return repeats

## 6. Konfigurasi dataset

`data.yaml` hanya berisi train dan valid. Test ditulis ke file terpisah (`data_test.yaml`) di bagian 10 dan 11, sehingga test tidak pernah terdaftar di konfigurasi training.

In [39]:
def write_data_yaml(path: Path, root: Path, val_split: str = "valid") -> Path:
    config = {
        "path": str(root),
        "train": "images/train",
        "val": f"images/{val_split}",
        "names": dict(enumerate(CLASS_NAMES)),
    }
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path

## 7. Training

Setiap run hanya berbeda satu hal dari baseline:

| Run | Model | Perbedaan |
|---|---|---|
| `E0_smoke` | YOLO11s | Uji alur: 2 epoch, 5% data |
| `A_yolo11s_s42` | YOLO11s | Baseline |
| `B_yolo26s_s42` | YOLO26s | Arsitektur |
| `C_yolo11s_bal_s42` | YOLO11s | Penyeimbangan kelas (repeat factor sampling) |
| `A800_yolo11s_s42` | YOLO11s | Lanjutan dari `best.pt` Run A pada resolusi 800 untuk lesi kecil |
| `A_fit_yolo11s_s42` | YOLO11s | Lanjutan dari `best.pt` Run A pada 640 dengan augmentasi berat dimatikan |

`A800_yolo11s_s42` tidak dimulai dari nol: bobot awalnya adalah hasil Run A milik tim sendiri, yang dilatih dari nol pada dataset resmi. Karena itu run ini memakai `pretrained=True`, yang di Ultralytics berarti *pakai bobot dari file `.pt` yang dimuat*; tanpa itu bobot Run A akan dibuang. Tidak ada bobot eksternal yang terlibat. Optimizer ditulis eksplisit (SGD, `lr0=0.002`) karena mode `auto` mengabaikan `lr0`, sedangkan fine-tuning butuh learning rate yang lebih kecil dari training awal. Run ini membutuhkan folder Run A (`RUNS_DIR/A_yolo11s_s42`) di Drive yang sama, misalnya lewat shortcut.

`A_fit_yolo11s_s42` juga melanjutkan dari `best.pt` Run A dengan aturan bobot dan optimizer yang sama, tetapi tetap pada resolusi 640. Evaluasi Run A pada sebagian data train menghasilkan F1 makro 0,744, jauh di bawah yang biasa dicapai model yang sudah menguasai data latihnya, sehingga Run A masih underfit. Run ini mematikan augmentasi yang paling banyak mengubah gambar (mosaic, mixup, variasi warna) supaya model belajar dari gambar train apa adanya. Flip horizontal/vertikal dan sedikit variasi skala serta pergeseran tetap aktif karena dataset sendiri memuat versi flip dan crop dari gambar yang sama. Data yang dipakai tetap hanya split train.

Output run ditulis ke Google Drive (`RUNS_DIR/RUN_ID`), termasuk checkpoint tiap 10 epoch. Kalau sesi Colab terputus, jalankan ulang notebook dengan `RUN_ID` yang sama: training dilanjutkan dari `last.pt`.

In [40]:
TRAIN_ARGS = dict(
    imgsz=IMGSZ,
    epochs=150,
    batch=16,
    seed=SEED,
    deterministic=True,
    pretrained=False,
    workers=2,
    cache=False,
    save_period=10,
    patience=0,
    close_mosaic=10,
    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    flipud=0.5,
    plots=True,
)

RUNS = {
    "E0_smoke": {"model": "yolo11s.yaml", "balanced": False, "overrides": {"epochs": 2, "fraction": 0.05}},
    "A_yolo11s_s42": {"model": "yolo11s.yaml", "balanced": False},
    "B_yolo26s_s42": {"model": "yolo26s.yaml", "balanced": False},
    "C_yolo11s_bal_s42": {"model": "yolo11s.yaml", "balanced": True, "repeat_threshold": 0.10},
    "A_fit_yolo11s_s42": {
        "model": RUNS_DIR / "A_yolo11s_s42" / "weights" / "best.pt",
        "balanced": False,
        "overrides": {"imgsz": 640, "epochs": 50, "pretrained": True, "optimizer": "SGD",
                      "lr0": 0.002, "lrf": 0.1, "warmup_epochs": 1.0,
                      "mosaic": 0.0, "mixup": 0.0, "close_mosaic": 0,
                      "hsv_h": 0.0, "hsv_s": 0.0, "hsv_v": 0.0,
                      "scale": 0.25, "translate": 0.05},
    },
    "A800_yolo11s_s42": {
        "model": RUNS_DIR / "A_yolo11s_s42" / "weights" / "best.pt",
        "balanced": False,
        "overrides": {"imgsz": 800, "epochs": 40, "pretrained": True, "optimizer": "SGD",
                      "lr0": 0.002, "lrf": 0.1, "warmup_epochs": 1.0},
    },
    "B800_yolo26s_s42": {
        "model": "yolo26s.yaml",
        "balanced": False,
        "overrides": {
            "imgsz": 800,
        },
    },
    "B_bal_yolo26s_s42": {
        "model": "yolo26s.yaml",
        "balanced": True,
        "repeat_threshold": 0.10,
    },
    "B_nomixup_yolo26s_s42": {
        "model": "yolo26s.yaml",
        "balanced": False,
        "overrides": {
            "mixup": 0.0,
        },
    },
}


def prepare_training_data(run: dict) -> Path:
    repeats = None
    if run["balanced"]:
        repeats = repeat_factors(train_images, train_boxes, run["repeat_threshold"], SEED)
    n_train = export_split("train", train_images, train_boxes, YOLO_DIR, repeats)
    n_valid = export_split("valid", valid_images, valid_boxes, YOLO_DIR)
    print(f"train: {n_train} gambar, valid: {n_valid} gambar")
    return write_data_yaml(WORK_DIR / "data.yaml", YOLO_DIR)


def training_finished(checkpoint: Path) -> bool:
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    return state.get("epoch", -1) == -1


def train(run_id: str) -> Path:
    run = RUNS[run_id]
    run_dir = RUNS_DIR / run_id
    last = run_dir / "weights" / "last.pt"
    data_yaml = prepare_training_data(run)

    if last.exists() and training_finished(last):
        print(f"{run_id} sudah selesai.")
    elif last.exists():
        print(f"Melanjutkan {run_id} dari {last}")
        YOLO(last).train(resume=True)
    else:
        if isinstance(run["model"], Path) and not run["model"].exists():
            raise FileNotFoundError(f"Bobot awal tidak ditemukan: {run['model']}")
        model = YOLO(str(run["model"]))
        model.train(data=str(data_yaml), project=str(RUNS_DIR), name=run_id, exist_ok=True,
                    **{**TRAIN_ARGS, **run.get("overrides", {})})
    return run_dir


if MODE == "full":
    run_dir = train(RUN_ID)

## 8. Evaluasi checkpoint di valid

Skor dihitung dalam dua versi:

- **E1**: rata-rata F1 per kelas dari `model.val()`. Ultralytics mengambil F1 pada satu confidence yang memaksimalkan rata-rata F1 semua kelas (IoU 0,5).
- **E2**: F1 makro 11 kelas pada confidence tetap `E2_CONF`, dihitung sendiri: prediksi diurutkan dari confidence tertinggi lalu dicocokkan secara greedy ke label dengan IoU ≥ 0,5 per kelas. Nilai `E2_CONF` dipilih dari valid.

mAP@50 ikut dicatat sebagai pembanding. Semua parameter evaluasi ditulis eksplisit (FP32, batch tetap). Setiap checkpoint dievaluasi pada resolusi saat ia dilatih (640, atau 800 untuk `A800_yolo11s_s42`), yang dibaca dari metadata checkpoint.

In [41]:
VAL_ARGS = dict(
    batch=EVAL_BATCH,
    conf=0.001,
    iou=0.7,
    max_det=300,
    quantize=32,
    plots=False,
    verbose=False,
    workers=2,
)


def trained_imgsz(model: YOLO) -> int:
    return int(
        model.overrides.get(
            "imgsz",
            IMGSZ,
        )
    )


def per_class_f1(metrics) -> np.ndarray:
    scores = np.full(
        len(CLASS_NAMES),
        np.nan,
    )

    scores[
        metrics.box.ap_class_index
    ] = metrics.box.f1

    return scores


def evaluate_e1(
    weights: Path,
    data_yaml: Path,
) -> dict:
    model = YOLO(weights)

    metrics = model.val(
        data=str(data_yaml),
        split="val",
        imgsz=trained_imgsz(model),
        **VAL_ARGS,
    )

    f1 = per_class_f1(metrics)

    return {
        "E1": float(np.nanmean(f1)),
        "mAP50": float(metrics.box.map50),
        **{
            f"F1 {name}": float(score)
            for name, score in zip(
                CLASS_NAMES,
                f1,
            )
        },
    }

In [42]:
def ground_truth(images: pd.DataFrame, boxes: pd.DataFrame) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    gt = {}
    grouped = dict(tuple(boxes.groupby("file_name")))
    for file_name in images["file_name"]:
        b = grouped.get(file_name)
        if b is None:
            gt[file_name] = (np.zeros(0, dtype=int), np.zeros((0, 4)))
        else:
            xyxy = np.stack([b["x"], b["y"], b["x"] + b["w"], b["y"] + b["h"]], axis=1)
            gt[file_name] = (b["class_id"].to_numpy(), xyxy)
    return gt


def predict_split(weights: Path, image_dir: Path, file_names: list[str]) -> dict[str, tuple]:
    model = YOLO(weights)
    preds = {}
    paths = [str(image_dir / name) for name in file_names]
    for start in range(0, len(paths), EVAL_BATCH):
        batch = paths[start:start + EVAL_BATCH]
        results = model.predict(batch, imgsz=trained_imgsz(model), conf=0.001, iou=0.7, max_det=300, quantize=32, verbose=False)
        for path, res in zip(batch, results):
            boxes = res.boxes
            preds[Path(path).name] = (boxes.cls.cpu().numpy().astype(int), boxes.conf.cpu().numpy(),
                                      boxes.xyxy.cpu().numpy())
    return preds


def box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    tl = np.maximum(a[:, None, :2], b[None, :, :2])
    br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(br - tl, 0, None), axis=2)
    area_a = np.prod(a[:, 2:] - a[:, :2], axis=1)
    area_b = np.prod(b[:, 2:] - b[:, :2], axis=1)
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def match_predictions(preds: dict, gt: dict, iou_threshold: float = 0.5):
    # Greedy matching per class in descending confidence. Truncating the sorted list at any
    # confidence threshold keeps the same matches, so one pass serves every threshold.
    classes, confs, hits = [], [], []
    n_gt = np.zeros(len(CLASS_NAMES), dtype=int)
    for file_name, (gt_cls, gt_xyxy) in gt.items():
        n_gt += np.bincount(gt_cls, minlength=len(CLASS_NAMES))
        p_cls, p_conf, p_xyxy = preds[file_name]
        for c in np.unique(p_cls):
            p_idx = np.where(p_cls == c)[0]
            p_idx = p_idx[np.argsort(-p_conf[p_idx], kind="stable")]
            g_idx = np.where(gt_cls == c)[0]
            matched = np.zeros(len(g_idx), dtype=bool)
            ious = box_iou(p_xyxy[p_idx], gt_xyxy[g_idx]) if len(g_idx) else np.zeros((len(p_idx), 0))
            for row, i in enumerate(p_idx):
                hit = False
                if len(g_idx):
                    candidates = np.where(~matched & (ious[row] >= iou_threshold))[0]
                    if len(candidates):
                        matched[candidates[np.argmax(ious[row, candidates])]] = True
                        hit = True
                classes.append(c)
                confs.append(p_conf[i])
                hits.append(hit)
    return np.array(classes, dtype=int), np.array(confs), np.array(hits, dtype=bool), n_gt


def macro_f1_at(matches, conf_threshold: float) -> tuple[float, np.ndarray]:
    classes, confs, hits, n_gt = matches
    keep = confs >= conf_threshold
    tp = np.bincount(classes[keep & hits], minlength=len(CLASS_NAMES))
    fp = np.bincount(classes[keep & ~hits], minlength=len(CLASS_NAMES))
    fn = n_gt - tp
    f1 = np.where(n_gt > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), np.nan)
    return float(np.nanmean(f1)), f1


def best_conf(matches, grid=np.round(np.arange(0.05, 0.951, 0.01), 2)) -> tuple[float, float]:
    scores = [macro_f1_at(matches, t)[0] for t in grid]
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])

def evaluate_checkpoint(
    weights: Path,
    data_yaml: Path,
    valid_gt: dict,
    valid_image_dir: Path,
) -> dict:
    if E2_CONF is None:
        raise ValueError(
            "E2_CONF harus diisi."
        )

    e1 = evaluate_e1(
        weights,
        data_yaml,
    )

    predictions = predict_split(
        weights,
        valid_image_dir,
        list(valid_gt),
    )

    matches = match_predictions(
        predictions,
        valid_gt,
    )

    e2_fixed, e2_f1 = macro_f1_at(
        matches,
        E2_CONF,
    )

    best_conf_value, e2_best = best_conf(
        matches,
    )

    e2_best_f1 = macro_f1_at(
        matches,
        best_conf_value,
    )[1]

    result = {
        **e1,
        "E2": float(e2_fixed),
        "E2_CONF": float(E2_CONF),
        "E2_best": float(e2_best),
        "E2_best_CONF": float(best_conf_value),
    }

    for name, score in zip(
        CLASS_NAMES,
        e2_f1,
    ):
        result[f"E2 F1 {name}"] = float(score)

    for name, score in zip(
        CLASS_NAMES,
        e2_best_f1,
    ):
        result[f"E2best F1 {name}"] = float(score)

    return result


def evaluate_run(
    run_id: str,
    data_yaml: Path,
    valid_gt: dict,
    valid_image_dir: Path,
) -> pd.DataFrame:
    weights_dir = (
        RUNS_DIR
        / run_id
        / "weights"
    )

    checkpoints = sorted(weights_dir.glob("epoch*.pt"),
        key=lambda path: int(
            path.stem[5:]
        ),
    )

    checkpoints += [
        weights_dir / name
        for name in (
            "last.pt",
            "best.pt",
        )
        if (weights_dir / name).exists()
    ]

    rows = []

    for checkpoint in checkpoints:
        print(
            f"Mengevaluasi {checkpoint.name}"
        )

        row = evaluate_checkpoint(
            checkpoint,
            data_yaml,
            valid_gt,
            valid_image_dir,
        )

        rows.append(
            {
                "run_id": run_id,
                "checkpoint": checkpoint.name,
                **row,
            }
        )

    table = pd.DataFrame(rows).sort_values(
        ["E1", "E2"],
        ascending=False,
    )

    output_path = (
        RUNS_DIR
        / run_id
        / "checkpoint_eval_valid.csv"
    )

    table.to_csv(
        output_path,
        index=False,
    )

    return table


if MODE == "full":
    valid_gt = ground_truth(
        valid_images,
        valid_boxes,
    )

    valid_table = evaluate_run(
        RUN_ID,
        WORK_DIR / "data.yaml",
        valid_gt,
        RAW_DIR / "valid",
    )

    display(valid_table.head(10))

## 9. Pemilihan model final

Kandidat diurutkan berdasarkan E1 di valid (tabel dari bagian 8, digabung dari semua run). Dari kandidat teratas dipilih yang kurva F1-nya landai di sekitar puncak dan tidak memiliki kelas dengan F1 sangat rendah. Split test tidak dilibatkan dalam keputusan ini.

Isi `SELECTED_WEIGHTS` dengan checkpoint terpilih. Sel ini menghitung `E2_CONF` dari valid; salin nilainya ke sel konfigurasi.

In [43]:
SELECTED_WEIGHTS = None  # model final v1.0: RUNS_DIR / "A_fit_yolo11s_s42" / "weights" / "epoch20.pt"

if MODE == "full" and SELECTED_WEIGHTS is not None:
    valid_gt = ground_truth(valid_images, valid_boxes)
    valid_preds = predict_split(Path(SELECTED_WEIGHTS), RAW_DIR / "valid", list(valid_gt))
    valid_matches = match_predictions(valid_preds, valid_gt)
    conf, score = best_conf(valid_matches)
    curve = pd.Series({t: macro_f1_at(valid_matches, t)[0] for t in (conf - 0.1, conf - 0.05, conf, conf + 0.05, conf + 0.1)})
    print(f"E2_CONF = {conf:.2f}  (F1 makro valid {score:.4f})")
    print("Kelandaian kurva F1 di sekitar puncak:")
    print(curve.round(4).to_string())

## 9b. Penyempurnaan bobot: model soup dan kalibrasi bias per kelas

Dua langkah ini tidak melatih ulang model dan hanya memakai split **valid** untuk mengambil keputusan. Hasilnya tetap satu file weights.

**Model soup (greedy).** Bobot EMA beberapa checkpoint dari satu run yang sama dirata-ratakan. Dimulai dari checkpoint terbaik, lalu checkpoint lain ditambahkan satu per satu dan hanya dipertahankan jika E1 valid naik.

**Kalibrasi bias per kelas.** Setiap kelas mencapai F1 terbaiknya pada confidence yang berbeda. Bias kelas pada lapisan klasifikasi head digeser sebesar `logit(t_bersama) − logit(t_kelas)`, sehingga titik terbaik tiap kelas jatuh di confidence yang sama (`E2_CONF`). Pergeseran untuk kelas dengan sedikit label ditarik ke arah nol dengan faktor `n / (n + 50)`. Karena tersimpan di dalam weights, efeknya terbawa ke evaluasi mana pun, tanpa threshold khusus. Kalibrasi ini adalah pemilihan 11 hyperparameter di valid, setara dengan memilih `E2_CONF`; tidak ada gradien yang dihitung dari data valid.

Hasil dipakai sebagai rilis baru hanya jika E1 valid naik cukup jelas dibanding checkpoint dasar.

In [44]:
from ultralytics.utils.torch_utils import strip_optimizer

REFINE_BASE = "epoch20.pt"
REFINE_CANDIDATES = ["epoch40.pt", "epoch10.pt", "epoch30.pt", "last.pt", "best.pt"]
CALIB_SHRINK = 50.0


def load_ema(path: Path) -> torch.nn.Module:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    return (ckpt.get("ema") or ckpt["model"]).float()


def average_checkpoints(paths: list[Path], dest: Path) -> Path:
    states = [load_ema(p).state_dict() for p in paths]
    averaged = {k: torch.stack([s[k] for s in states]).mean(0) if v.is_floating_point() else v
                for k, v in states[0].items()}
    ckpt = torch.load(paths[0], map_location="cpu", weights_only=False)
    model = load_ema(paths[0])
    model.load_state_dict(averaged)
    ckpt["ema"], ckpt["model"] = model, None
    torch.save(ckpt, dest)
    strip_optimizer(dest)
    return dest


def greedy_soup(weights_dir: Path, base: str, candidates: list[str], data_yaml: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    soup_path, trial_path = out_dir / "soup.pt", out_dir / "soup_trial.pt"
    chosen = [base]
    best = evaluate_e1(average_checkpoints([weights_dir / base], soup_path), data_yaml)["E1"]
    rows = [{"checkpoint": base, "E1": best, "diterima": True}]
    for name in candidates:
        if not (weights_dir / name).exists():
            continue
        score = evaluate_e1(average_checkpoints([weights_dir / n for n in chosen + [name]], trial_path), data_yaml)["E1"]
        accepted = score > best
        rows.append({"checkpoint": name, "E1": score, "diterima": accepted})
        if accepted:
            chosen.append(name)
            best = score
            shutil.copy2(trial_path, soup_path)
    trial_path.unlink(missing_ok=True)
    return soup_path, chosen, pd.DataFrame(rows)


def logit(p: float) -> float:
    return math.log(p / (1 - p))


def class_bias_offsets(matches, target_conf: float, shrink: float) -> np.ndarray:
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
    curves = np.array([macro_f1_at(matches, t)[1] for t in grid])
    n_gt = matches[3]
    offsets = np.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if n_gt[c] == 0:
            continue
        best_t = float(grid[int(np.nanargmax(curves[:, c]))])
        offsets[c] = (logit(target_conf) - logit(best_t)) * n_gt[c] / (n_gt[c] + shrink)
    return offsets


def apply_class_bias(weights: Path, offsets: np.ndarray, dest: Path) -> Path:
    ckpt = torch.load(weights, map_location="cpu", weights_only=False)
    model = ckpt["model"].float()
    head = model.model[-1]
    branches = [head.cv3] + ([head.one2one_cv3] if hasattr(head, "one2one_cv3") else [])
    with torch.no_grad():
        for branch in branches:
            for seq in branch:
                seq[-1].bias += torch.as_tensor(offsets, dtype=seq[-1].bias.dtype)
    ckpt["model"] = model.half()
    torch.save(ckpt, dest)
    return dest


def e2_at(weights: Path, gt: dict, conf: float) -> tuple[float, tuple]:
    matches = match_predictions(predict_split(weights, RAW_DIR / "valid", list(gt)), gt)
    return macro_f1_at(matches, conf)[0], matches


if MODE == "full" and REFINE_WEIGHTS:
    target_conf = E2_CONF if E2_CONF is not None else 0.26
    data_yaml = WORK_DIR / "data.yaml"
    out_dir = RUNS_DIR / f"{RUN_ID}_refined"
    valid_gt = ground_truth(valid_images, valid_boxes)

    soup_path, chosen, soup_table = greedy_soup(RUNS_DIR / RUN_ID / "weights", REFINE_BASE,
                                                REFINE_CANDIDATES, data_yaml, out_dir)
    print("Checkpoint dalam soup:", chosen)
    display(soup_table.round(4))

    soup_e2, soup_matches = e2_at(soup_path, valid_gt, target_conf)
    offsets = class_bias_offsets(soup_matches, target_conf, CALIB_SHRINK)
    calibrated = apply_class_bias(soup_path, offsets, out_dir / "soup_calibrated.pt")
    calib_e1 = evaluate_e1(calibrated, data_yaml)
    calib_e2, _ = e2_at(calibrated, valid_gt, target_conf)

    summary = pd.DataFrame([
        {"model": f"{REFINE_BASE} (dasar)", "E1": soup_table.iloc[0]["E1"]},
        {"model": "soup", "E1": soup_table.loc[soup_table["diterima"], "E1"].max(), f"E2@{target_conf}": soup_e2},
        {"model": "soup + kalibrasi", "E1": calib_e1["E1"], f"E2@{target_conf}": calib_e2},
    ])
    display(summary.round(4))
    display(pd.Series(offsets, index=CLASS_NAMES, name="geseran bias").round(3))
    (out_dir / "refine_summary.json").write_text(json.dumps({
        "soup": chosen, "offsets": dict(zip(CLASS_NAMES, offsets.round(4).tolist())),
        "summary": summary.round(4).to_dict(orient="records")}, indent=2))
    print(f"Weights hasil: {calibrated}")

## 10. Evaluasi test (sekali)

Dijalankan **satu kali** setelah model final ditetapkan, dengan mengubah `RUN_FINAL_TEST = True`. Label test hanya dipetakan ke 11 kelas dan dipotong ke batas gambar, tanpa pembersihan lain.

In [45]:
def prepare_test_data() -> tuple[Path, dict]:
    test_images, test_boxes = frames["test"][0], clip_boxes(frames["test"][1])
    export_split("test", test_images, test_boxes, YOLO_DIR)
    data_yaml = write_data_yaml(WORK_DIR / "data_test.yaml", YOLO_DIR, val_split="test")
    return data_yaml, ground_truth(test_images, test_boxes)


def score_test(weights: Path, e2_conf: float) -> dict:
    data_yaml, test_gt = prepare_test_data()
    e1 = evaluate_e1(weights, data_yaml)
    matches = match_predictions(predict_split(weights, RAW_DIR / "test", list(test_gt)), test_gt)
    e2, _ = macro_f1_at(matches, e2_conf)
    return {"E1": e1["E1"], "E2": e2, "E2_CONF": e2_conf, "mAP50": e1["mAP50"],
            **{k: v for k, v in e1.items() if k.startswith("F1 ")}}


if MODE == "full" and RUN_FINAL_TEST:
    assert SELECTED_WEIGHTS is not None and E2_CONF is not None, "Tetapkan model final dan E2_CONF lebih dulu."
    test_scores = score_test(Path(SELECTED_WEIGHTS), E2_CONF)
    (RUNS_DIR / "final_test_scores.json").write_text(json.dumps(test_scores, indent=2))
    display(pd.Series(test_scores).round(4))

## 11. Inference dari weights rilis

Jalur yang dipakai untuk audit: weights diunduh dari GitHub Release, checksum diverifikasi, lalu skor dihitung pada split test resmi. Bagian ini tidak bergantung pada hasil bagian 4 sampai 10.

In [46]:
def download_weights(url: str, expected_sha256: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    actual = sha256sum(dest)
    if actual != expected_sha256:
        raise ValueError(f"Checksum weights tidak cocok: {actual}")
    return dest


if MODE == "inference_only":
    assert WEIGHTS_URL and WEIGHTS_SHA256 and E2_CONF is not None, "Isi WEIGHTS_URL, WEIGHTS_SHA256, dan E2_CONF."
    started = time.perf_counter()
    weights = download_weights(WEIGHTS_URL, WEIGHTS_SHA256, WORK_DIR / "weights" / Path(WEIGHTS_URL).name)
    scores = score_test(weights, E2_CONF)
    scores["seconds"] = round(time.perf_counter() - started, 1)
    display(pd.Series(scores).round(4))